# Data Augmentation

# Import Libraries

In [ ]:
# !pip install datasets
# !pip install sentence_transformers
# !pip install jsonlines

In [3]:
import numpy as np
import pandas as pd
from datasets import load_dataset
import time
import random
import jsonlines
import re

from sentence_transformers import CrossEncoder
from transformers import pipeline

# Import Dataset

In [ ]:
fever_data = load_dataset("tommasobonomo/sem_augmented_fever_nli")

In [5]:
fever_data

DatasetDict({
    train: Dataset({
        features: ['id', 'premise', 'hypothesis', 'label', 'wsd', 'srl'],
        num_rows: 51086
    })
    validation: Dataset({
        features: ['id', 'premise', 'hypothesis', 'label', 'wsd', 'srl'],
        num_rows: 2288
    })
    test: Dataset({
        features: ['id', 'premise', 'hypothesis', 'label', 'wsd', 'srl'],
        num_rows: 2287
    })
})

# Adversarial Examples

In [6]:
def save_jsonl(results, file_name):
  with jsonlines.open(file_name, mode = 'w') as writer:
    for result in results:
      writer.write(result)

def read_jsonl(file_name, num):
  with jsonlines.open(file_name) as reader:
    for i, line in enumerate(reader):
      if i < num:
        print("Premise: ", line['premise'])
        print("Hypothesis: ", line['hypothesis'])
        print("Label: ", line['label'], "\n")
      else:
        break

In [7]:
def test_adversarial_examples(model, hypothesis_list, num):
  adversarial_examples = []
  for dic in hypothesis_list[:num]:
    premise = dic['premise']
    hypo = dic['hypothesis']
    adversarial_examples.append((premise, hypo))

  scores = model.predict(adversarial_examples)
  label_mapping = ['contradiction', 'entailment', 'neutral']
  labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]

  confidences = scores.max(axis=1)
  results = list(zip(labels, confidences))
  return results

In [ ]:
# Upload the DeBERTA V3 model trained on the NLI task to assess the quality of the results
pre_trained_NLI_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

In [ ]:
# Upload GPT2 for task generation purposes
text_generation_gpt2 = pipeline("text-generation", model = "gpt2", pad_token_id = 0)

In [10]:
# Indices corresponding to 'entailment' labels
entailment_indices = [i for i, label in enumerate(fever_data['train']['label']) if label == 'ENTAILMENT']

# Indices corresponding to 'contradiction' labels
contradiction_indices = [i for i, label in enumerate(fever_data['train']['label']) if label == 'CONTRADICTION']

## Neutral Adversarial Examples

### First approach: Swap hypothesis

In [11]:
# Set a seed for riproducibility
np.random.seed(0)

# Select random indices from both 'entailment' and 'contradiction' indices
random_indices_entailment = np.random.choice(entailment_indices, 5000, replace=False)
random_indices_contradiction = np.random.choice(contradiction_indices, 5000, replace=False)
random_indices_1 = np.concatenate([random_indices_entailment, random_indices_contradiction])
random_indices_1[:10]

array([ 2296, 19026, 42166,   673, 17242, 26685, 30045, 38453, 48102,
        9803])

In [12]:
def swap_hypothesis(premise, entailment_hypothesis, i, last_index):
  if i < last_index:
      neutral_hypothesis = fever_data['train']['hypothesis'][random_indices_1[i+1]]
  else:
       neutral_hypothesis = fever_data['train']['hypothesis'][random_indices_1[0]]

  return {
        'premise': premise,
        'hypothesis': neutral_hypothesis,
        'label' : 'NEUTRAL'
    }

In [15]:
start_time = time.time()
neutral_hypothesis_list = []
last_index = len(random_indices_1) - 1

for i in random_indices_1:
  premise = fever_data['train']['premise'][i]
  entailment_hypothesis = fever_data['train']['hypothesis'][i]
  result = swap_hypothesis(premise, entailment_hypothesis, i, last_index)

  # Check the quality of the results using DebertV3 model trained on NLI task
  scores = pre_trained_NLI_model.predict([(result['premise'], result['hypothesis'])])
  label_mapping = ['contradiction', 'entailment', 'neutral']
  labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]

  if labels[0] == 'neutral':
    neutral_hypothesis_list.append(result)
    num = len(neutral_hypothesis_list)
    if num % 100 == 0:
      print(f"Generated {num} adversarial neutral examples")

end_time = time.time()
print("Execution time: ", round((end_time - start_time)/60, 3), "minutes.")

Generated 100 adversarial neutral examples
Generated 200 adversarial neutral examples
Generated 300 adversarial neutral examples
Generated 400 adversarial neutral examples
Generated 500 adversarial neutral examples
Generated 600 adversarial neutral examples
Generated 700 adversarial neutral examples
Generated 800 adversarial neutral examples
Generated 900 adversarial neutral examples
Generated 1000 adversarial neutral examples
Generated 1100 adversarial neutral examples
Generated 1200 adversarial neutral examples
Generated 1300 adversarial neutral examples
Generated 1400 adversarial neutral examples
Generated 1500 adversarial neutral examples
Generated 1600 adversarial neutral examples
Generated 1700 adversarial neutral examples
Generated 1800 adversarial neutral examples
Generated 1900 adversarial neutral examples
Generated 2000 adversarial neutral examples
Generated 2100 adversarial neutral examples
Generated 2200 adversarial neutral examples
Generated 2300 adversarial neutral exampl

In [16]:
print("Examples evaluation using DeBERTaV3 fine-tuned on NLI: \n")
test_adversarial_examples(pre_trained_NLI_model, neutral_hypothesis_list, 10)

Examples evaluation using DeBERTaV3 fine-tuned on NLI: 



[('neutral', 5.4552345),
 ('neutral', 2.9491808),
 ('neutral', 4.282059),
 ('neutral', 4.312654),
 ('neutral', 5.4593124),
 ('neutral', 5.511794),
 ('neutral', 5.3541527),
 ('neutral', 6.04193),
 ('neutral', 5.170422),
 ('neutral', 4.528471)]

In [17]:
save_jsonl(neutral_hypothesis_list, 'neutral_adversarial_examples_first_approach.jsonl')

In [18]:
read_jsonl('neutral_adversarial_examples_first_approach.jsonl', 5)

Premise:  Morgan Freeman ( born June 1 , 1937 ) is an American actor , producer and narrator . Freeman won an Academy Award in 2005 for Best Supporting Actor with Million Dollar Baby ( 2004 ) , and he has received Oscar nominations for his performances in Street Smart ( 1987 ) , Driving Miss Daisy ( 1989 ) , The Shawshank Redemption ( 1994 ) and Invictus ( 2009 ) . He has also won a Golden Globe Award and a Screen Actors Guild Award . Morgan Freeman is ranked as the 4th highest box office star with over $ 4.316 billion total box office gross , an average of $ 74.4 million per film .
Hypothesis:  Russell Crowe has worked with Ridley Scott.
Label:  NEUTRAL 

Premise:  Michael Jackson . In 1993 , he was accused of child sexual abuse , but the civil case was settled out of court for an undisclosed amount and no formal charges were brought . In 2005 , he was tried and acquitted of further child sexual abuse allegations and several other charges after the jury found him not guilty on all cou

### Second Approach: Generate Neutral Text with GPT2


In [23]:
# Set a seed for riproducibility
np.random.seed(1)

# Select random indices from both 'entailment' and 'contradiction' indices
random_indices_entailment = np.random.choice(entailment_indices, 250, replace=False)
random_indices_contradiction = np.random.choice(contradiction_indices, 250, replace=False)
random_indices_2 = np.concatenate([random_indices_entailment, random_indices_contradiction])
random_indices_2[:10]

array([26298, 45628, 42553,  3000, 36061,  3553, 22064, 50285, 10070,
        3893])

In [24]:
def generate_neutral_hypothesis(premise, entailment_hypothesis):
  # Define the prompt for text generation using GPT2 model
  prompt = f"Premise: {premise} Neutral Hypothesis:"
  generated_text = text_generation_gpt2(prompt, max_new_tokens = 30, num_return_sequences = 1)[0]['generated_text']
  neutral_hypothesis = generated_text.split("Neutral Hypothesis:", 1)[-1].strip()

  # Prevent subsequent sentences from being truncated prematurely (max_new_tokens = 30)
  last_period = neutral_hypothesis.rfind(".")
  if last_period != -1:
      neutral_hypothesis = neutral_hypothesis[:last_period + 1]
  else:
      neutral_hypothesis = neutral_hypothesis

  return {
        'premise': premise,
        'hypothesis': neutral_hypothesis,
        'label' : 'NEUTRAL'
    }

In [25]:
start_time = time.time()
neutral_hypothesis_list_2 = []
for i in random_indices_2:
  premise = fever_data['train']['premise'][i]
  entailment_hypothesis = fever_data['train']['hypothesis'][i]
  neutral_hypothesis = generate_neutral_hypothesis(premise, entailment_hypothesis)

  # Check the quality of the results using DebertV3 model trained on NLI task
  scores = pre_trained_NLI_model.predict([(neutral_hypothesis['premise'], neutral_hypothesis['hypothesis'])])
  label_mapping = ['contradiction', 'entailment', 'neutral']
  labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]

  if labels[0] == 'neutral':
    neutral_hypothesis_list_2.append(neutral_hypothesis)
    num = len(neutral_hypothesis_list_2)
    if num % 100 == 0:
        print(f"Generated {num} adversarial neutral examples")

end_time = time.time()
print("Execution time: ", round((end_time - start_time)/60, 3), "minutes.")

Generated 100 adversarial neutral examples
Generated 200 adversarial neutral examples
Generated 300 adversarial neutral examples
Execution time:  18.371 minutes.


In [26]:
print("Examples evaluation using DeBERTaV3 fine-tuned on NLI: \n")
test_adversarial_examples(pre_trained_NLI_model, neutral_hypothesis_list_2, 10)

Examples evaluation using DeBERTaV3 fine-tuned on NLI: 



[('neutral', 4.2264447),
 ('neutral', 3.5070016),
 ('neutral', 5.5017858),
 ('neutral', 5.6926513),
 ('neutral', 2.865475),
 ('neutral', 4.8079233),
 ('neutral', 5.4515543),
 ('neutral', 5.4591727),
 ('neutral', 4.8554783),
 ('neutral', 5.1676574)]

In [27]:
save_jsonl(neutral_hypothesis_list_2, 'neutral_adversarial_examples_second_approach.jsonl')

In [28]:
read_jsonl('neutral_adversarial_examples_second_approach.jsonl', 5)

Premise:  Daniel Richard McBride ( born December 29 , 1976 ) is an American actor , writer , and comedian . James Edward Franco ( born April 19 , 1978 ) is an American actor and filmmaker known for his work in both comedic and dramatic films and TV shows . For his role in 127 Hours ( 2010 ) , Franco was nominated for an Academy Award for Best Actor . Justin Paul Theroux ( born August 10 , 1971 ) is an American actor and screenwriter . Your Highness is a 2011 American stoner comic fantasy film directed by David Gordon Green , and stars Danny McBride , James Franco , Natalie Portman , Zooey Deschanel and Justin Theroux .
Hypothesis:  This review is based (and probably written and directed) on a screenplay that was originally written by a writer named Richard McBride and directed by Gerard Way
Label:  NEUTRAL 

Premise:  Ray Milland . After being released by MGM , he was picked up by Paramount , who used Milland in a range of lesser speaking parts , normally as an English character . Mill

## Contradiction Adversarial Examples

### First Approach: Introduce negation in the hypothesis

In [33]:
# Set a seed for riproducibility
np.random.seed(2)

# Select random indices from 'entailment' indices
random_indices_3 = np.random.choice(entailment_indices, 1000, replace=False)
random_indices_3[:10]

array([47132, 34375, 34891, 41231, 31295, 42703, 10857, 22139, 45035,
        1044])

In [34]:
def introduce_negation(premise, entailment_hypothesis, negation):
  contradiction_hypothesis = negation + ' ' + entailment_hypothesis
  return {
        'premise': premise,
        'hypothesis': contradiction_hypothesis,
        'label' : 'CONTRADICTION'
    }

In [35]:
start_time = time.time()
contradiction_hypothesis_list = []
negations = ['It is not the case that', 'Not necessary', 'It is not true that', 'In contrast to']

for i in random_indices_3:
  negation = np.random.choice(negations, 1)[0]
  premise = fever_data['train']['premise'][i]
  entailment_hypothesis = fever_data['train']['hypothesis'][i]
  contradiction_hypothesis = introduce_negation(premise, entailment_hypothesis, negation)

  # Check the quality of the results using DebertV3 model trained on NLI task
  scores = pre_trained_NLI_model.predict([(contradiction_hypothesis['premise'], contradiction_hypothesis['hypothesis'])])
  label_mapping = ['contradiction', 'entailment', 'neutral']
  labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]

  if labels[0] == 'contradiction':
    contradiction_hypothesis_list.append(contradiction_hypothesis)
    num = len(contradiction_hypothesis_list)
    if num % 100 == 0:
      print(f"Generated {num} contradiction examples")

end_time = time.time()
print("Execution time:", round((end_time - start_time)/60, 3), "minutes.")

Generated 100 contradiction examples
Generated 200 contradiction examples
Generated 300 contradiction examples
Generated 400 contradiction examples
Generated 500 contradiction examples
Generated 600 contradiction examples
Generated 700 contradiction examples
Execution time: 3.609 minutes.


In [36]:
print("Examples evaluation using DeBERTaV3 fine-tuned on NLI: \n")
test_adversarial_examples(pre_trained_NLI_model, contradiction_hypothesis_list, 10)

Examples evaluation using DeBERTaV3 fine-tuned on NLI: 



[('contradiction', 3.2954879),
 ('contradiction', 2.5758264),
 ('contradiction', 6.3978195),
 ('contradiction', 4.743758),
 ('contradiction', 0.42350128),
 ('contradiction', 1.2092559),
 ('contradiction', 2.6065207),
 ('contradiction', 1.3970472),
 ('contradiction', 4.3272305),
 ('contradiction', 5.623229)]

In [37]:
save_jsonl(contradiction_hypothesis_list, 'contradiction_adversarial_examples_first_approach.jsonl')

In [38]:
read_jsonl('contradiction_adversarial_examples_first_approach.jsonl', 5)

Premise:  Macbeth . It was first published in the Folio of 1623 , possibly from a prompt book , and is Shakespeare 's shortest tragedy .
Hypothesis:  In contrast to Shakespeare's shortest tragedy is Macbeth.
Label:  CONTRADICTION 

Premise:  Media circus is a colloquial metaphor , or idiom , describing a news event where the level of media coverage - measured by such factors as the number of reporters at the scene and the amount of material broadcast or published - is perceived to be excessive or out of proportion to the event being covered . Olympic Games . The growing importance of mass media created the issue of corporate sponsorship and commercialisation of the Games . Every two years the Olympics and its media exposure provide unknown athletes with the chance to attain national and sometimes international fame .
Hypothesis:  It is not the case that There is media coverage of the Olympic Games.
Label:  CONTRADICTION 

Premise:  Sheldon Haley ( born December 18 , 1972 ) , better kno

### Second Approach: Swap Agent-Patient using Semantic Role Labeling (SRL)

In [45]:
# Set a seed for riproducibility
np.random.seed(3)

# Select random indices from 'entailment' indices
random_indices_4 = np.random.choice(entailment_indices, 10000, replace=False)
random_indices_4[:10]

array([43860, 29140, 21223, 33198, 35299, 39659, 39061, 14302,  3827,
       24186])

In [46]:
# Example of the Semantic Role Labeling (SRL) dictionary in the dataset
fever_data['train']['srl'][random_indices_4[0]]['hypothesis']

{'tokens': [{'index': 0, 'rawText': 'Oliver'},
  {'index': 1, 'rawText': 'Reed'},
  {'index': 2, 'rawText': 'was'},
  {'index': 3, 'rawText': 'in'},
  {'index': 4, 'rawText': 'Castaway'},
  {'index': 5, 'rawText': '.'}],
 'annotations': [{'tokenIndex': 2,
   'verbatlas': {'frameName': 'COPULA',
    'roles': [{'role': 'Theme', 'score': 1.0, 'span': [0, 2]},
     {'role': 'Attribute', 'score': 1.0, 'span': [3, 5]}]},
   'englishPropbank': {'frameName': 'be.01',
    'roles': [{'role': 'ARG1', 'score': 1.0, 'span': [0, 2]},
     {'role': 'ARG2', 'score': 1.0, 'span': [3, 5]}]}}]}

In [47]:
def extract_roles_from_annotations(annotations):
  agents = []
  patients = []

  for annotation in annotations:
    if 'englishPropbank' in annotation:  # EnglishProbank access
      englishPropbank_roles = annotation['englishPropbank']['roles']
      for role in englishPropbank_roles:
        if role['role'] == 'ARG0':  # ARG0 -> subject of the sentence
          agents.append(role['span'])
        elif role['role'] == 'ARG1':  # ARG 1 -> object of the sentence
          patients.append(role['span'])

  return agents, patients

In [48]:
def swap_agent_patient(hypothesis, agents, patients):
  idx0, idx1 = agents[0][0], agents[0][1]
  idx2, idx3 = patients[0][0], patients[0][1]

  # Identify agent (ARG0) and patient (ARG1)
  agent = ' '.join(hypothesis[idx0:idx1])
  patient = ' '.join(hypothesis[idx2:idx3])

  # Remove the punctuation at the end of patient phrase for consistency
  patient = patient[:-1]

  # Swap Agent-Patient and reconstruct the sentence
  swapped_hypothesis = hypothesis[:idx0] + patient.split() + hypothesis[idx1:idx2] + agent.split() + hypothesis[idx3:]
  swapped_hypothesis[0] = swapped_hypothesis[0].capitalize()
  swapped_hypothesis = ' '.join(swapped_hypothesis)
  swapped_hypothesis += '.'

  return swapped_hypothesis

In [49]:
start_time = time.time()

contradiction_hypothesis_list_2 = []
srl_data = fever_data['train']['srl']
hypothesis_data = fever_data['train']['hypothesis']

for i in random_indices_4:
  agents = []
  patients = []
  annotations = srl_data[i]['hypothesis']['annotations']
  agents, patients = extract_roles_from_annotations(annotations)

  if agents and patients and agents[0][0] < patients[0][0]:
    agent_patient_swap = {'premise': '', 'hypothesis': '', 'label' : 'CONTRADICTION'}
    hypothesis = hypothesis_data[i].split()
    original_hypothesis = ' '.join(hypothesis)

    premise = fever_data['train']['premise'][i]
    agent_patient_swap['premise'] = premise

    swapped_hypothesis = swap_agent_patient(hypothesis, agents, patients)
    agent_patient_swap['hypothesis'] = swapped_hypothesis

    # Check the quality of the results using DebertV3 model trained on NLI task
    scores = pre_trained_NLI_model.predict([(agent_patient_swap['premise'], agent_patient_swap['hypothesis'])])
    label_mapping = ['contradiction', 'entailment', 'neutral']
    labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]

    if labels[0] == 'contradiction':
      contradiction_hypothesis_list_2.append(agent_patient_swap)
      num = len(contradiction_hypothesis_list_2)
      if num % 100 == 0:
        print(f"Generated {num} adversarial contradiction examples")

end_time = time.time()
print("Execution time:", round((end_time - start_time) / 60, 3), "minutes")

Generated 100 adversarial contradiction examples
Generated 200 adversarial contradiction examples
Generated 300 adversarial contradiction examples
Execution time: 5.152 minutes


In [51]:
print("Examples evaluation using DeBERTaV3 fine-tuned on NLI: \n")
test_adversarial_examples(pre_trained_NLI_model, contradiction_hypothesis_list_2, 10)

Examples evaluation using DeBERTaV3 fine-tuned on NLI: 



[('contradiction', 1.7541236),
 ('contradiction', 7.484199),
 ('contradiction', 2.378491),
 ('contradiction', 4.7838144),
 ('contradiction', 6.2922025),
 ('contradiction', 3.6647623),
 ('contradiction', 4.4069014),
 ('contradiction', 3.4651797),
 ('contradiction', 3.9932745),
 ('contradiction', 4.160154)]

In [52]:
save_jsonl(contradiction_hypothesis_list_2, 'contradiction_adversarial_examples_second_approach.jsonl')

In [53]:
read_jsonl('contradiction_adversarial_examples_second_approach.jsonl', 5)

Premise:  Asceticism ( [ əˈsɛtɪsɪzəm ] from the [ Wiktionary : ἄσκησις , ἄσκησις ] áskesis , `` exercise '' or `` training '' ) is a lifestyle characterized by abstinence from worldly pleasures , often for the purpose of pursuing spiritual goals . The practitioners of these religions eschewed worldly pleasures and led an abstinent lifestyle , in the pursuit of redemption , salvation or spirituality .
Hypothesis:  Pleasures eschew Followers of asceticism.
Label:  CONTRADICTION 

Premise:  Tom Hardy . He has appeared in three Christopher Nolan films : the science fiction thriller Inception ( 2010 ) , portrayed Bane in the superhero film The Dark Knight Rises ( 2012 ) , and the action-thriller Dunkirk ( 2017 ) based on the British evacuation in World War II .
Hypothesis:  To appear in Dunkir refused Tom Hardy (2017)..
Label:  CONTRADICTION 

Premise:  La Vie en rose (film) . La Môme ) La Môme refers to Piaf 's nickname `` La Môme Piaf '' ( meaning `` baby sparrow , birdie , little

### Third Approach: Numerical Inference

In [54]:
# Set a seed for riproducibility
np.random.seed(4)

# Select random indices from 'entailment' indices
random_indices_5 = np.random.choice(entailment_indices, 10000, replace=False)
random_indices_5[:10]

array([48690, 25584, 41528, 28800, 27295, 24484,  2284, 23001, 42416,
       30264])

In [55]:
def apply_numerical_inference(premise, hypo):
  incremented_number = None
  num_found = False

  # Find year using the pattern
  year_pattern = r'\b\d{4}\b'

  match = re.search(year_pattern, premise)
  if match:
      num_found = True
      year = int(match.group())
      # Increment the year by 1 to introduce contradiction
      incremented_year = year + 1
      incremented_number = str(incremented_year)

  # Reconstruct the sentence
  if num_found and incremented_number is not None:
    contradiction_hypothesis = hypo[:-1] + ' in ' + incremented_number + '.'
    return {
            'premise': premise,
            'hypothesis': contradiction_hypothesis,
            'label': 'CONTRADICTION'
        }
  else:
      return None

In [58]:
start_time = time.time()
contradiction_hypothesis_list_3 = []
for i in random_indices_5:
  premise = fever_data['train']['premise'][i]
  entailment_hypothesis = fever_data['train']['hypothesis'][i]

  result = apply_numerical_inference(premise, entailment_hypothesis)
  if result:
    # Check the quality of the results using DebertV3 model trained on NLI task
    scores = pre_trained_NLI_model.predict([(result['premise'], result['hypothesis'])])
    label_mapping = ['contradiction', 'entailment', 'neutral']
    labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]

    if labels[0] == 'contradiction':
      contradiction_hypothesis_list_3.append(result)
      num = len(contradiction_hypothesis_list_3)
      if num % 100 == 0:
        print(f"Generated {num} adversarial contradiction examples")

end_time = time.time()
print("Execution time: ", round((end_time - start_time)/60, 3), "minutes.")

Generated 100 adversarial contradiction examples
Generated 200 adversarial contradiction examples
Generated 300 adversarial contradiction examples
Generated 400 adversarial contradiction examples
Generated 500 adversarial contradiction examples
Generated 600 adversarial contradiction examples
Generated 700 adversarial contradiction examples
Generated 800 adversarial contradiction examples
Generated 900 adversarial contradiction examples
Generated 1000 adversarial contradiction examples
Generated 1100 adversarial contradiction examples
Generated 1200 adversarial contradiction examples
Generated 1300 adversarial contradiction examples
Generated 1400 adversarial contradiction examples
Generated 1500 adversarial contradiction examples
Generated 1600 adversarial contradiction examples
Generated 1700 adversarial contradiction examples
Generated 1800 adversarial contradiction examples
Generated 1900 adversarial contradiction examples
Generated 2000 adversarial contradiction examples
Generated

In [59]:
print("Examples evaluation using DeBERTaV3 fine-tuned on NLI: \n")
test_adversarial_examples(pre_trained_NLI_model, contradiction_hypothesis_list_3, 10)

Examples evaluation using DeBERTaV3 fine-tuned on NLI: 



[('contradiction', 1.6377085),
 ('contradiction', 2.6806479),
 ('contradiction', 5.2595496),
 ('contradiction', 5.5665135),
 ('contradiction', 0.61185443),
 ('contradiction', 0.669656),
 ('contradiction', 2.7073278),
 ('contradiction', 2.178851),
 ('contradiction', 3.237176),
 ('contradiction', 4.6565943)]

In [60]:
save_jsonl(contradiction_hypothesis_list_3, 'contradiction_adversarial_examples_third_approach.jsonl')

In [61]:
read_jsonl('contradiction_adversarial_examples_third_approach.jsonl', 5)

Premise:  21 is the second studio album by British singer Adele . Adele Laurie Blue Adkins ( [ əˈdɛl ] born 5 May 1988 ) is an English singer-songwriter . Her third concert tour , Adele Live 2016 , visited Europe , North America and Oceania , and will conclude with four finale concerts at Wembley Stadium in mid-2017 . Adele Live 2016 ( titled as Adele Live 2017 for the shows in 2017 ) is the third concert tour by British singer Adele in support of her third studio album , 25 . `` Skyfall '' is the theme song of the 2012 James Bond film Skyfall , performed by British singer Adele .
Hypothesis:  Adele sings in 1989.
Label:  CONTRADICTION 

Premise:  Macbeth ( or The Tragedy of Macbeth ) is a 1971 British-American historical drama film directed by Roman Polanski and co-written by Polanski and Kenneth Tynan . It was somewhat controversial for its depictions of graphic violence and nudity , but has also received positive attention , and was named Best Film by the National Board of Review . 